# Orchestrator Agent with Sub-Agents

This demo creates two specialist Microsoft Foundry agents and an orchestrator agent. The orchestrator receives the specialists' outputs and produces one implementation-ready answer.

In [1]:
%pip install -q azure-ai-projects==2.0.0b2 azure-identity python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Load the existing project configuration

The notebook reuses the current details from `A2A/A2A_and_MCP/.env`.

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from azure.ai.projects.aio import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity.aio import DefaultAzureCredential

env_path = (Path.cwd().parent / "A2A_and_MCP" / ".env").resolve()
load_dotenv(env_path)

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

assert foundry_project_endpoint, f"FOUNDRY_PROJECT_ENDPOINT was not found in {env_path}"
assert model_deployment_name, f"MODEL_DEPLOYMENT_NAME was not found in {env_path}"

print(f"Loaded configuration from: {env_path}")
print(f"Model deployment: {model_deployment_name}")

Loaded configuration from: D:\TRAININGS_Recent_Sessions\EY_Agentic_AI_Level3-May-June2026\udmy\MicrosoftAI-Foundry-main\A2A\A2A_and_MCP\.env
Model deployment: ajay-gpt-4o


## Connect to Foundry and create the agent team

In [3]:
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=foundry_project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

async def create_prompt_agent(name: str, instructions: str):
    agent = await project_client.agents.create_version(
        agent_name=name,
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions=instructions,
        ),
    )
    print(f"Created {agent.name} version {agent.version}")
    return agent

async def invoke_agent(agent, prompt: str) -> str:
    conversation = await openai_client.conversations.create()
    response = await openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
        input=prompt,
    )
    return response.output_text

requirements_agent = await create_prompt_agent(
    "orchestration-requirements-agent",
    "You are a requirements analyst. Identify goals, assumptions, constraints, risks, and acceptance criteria. Be concise and concrete.",
)

implementation_agent = await create_prompt_agent(
    "orchestration-implementation-agent",
    "You are a senior Azure implementation specialist. Produce practical implementation steps, commands, security guidance, and verification checks.",
)

orchestrator_agent = await create_prompt_agent(
    "orchestration-manager-agent",
    "You are the orchestrator. Synthesize specialist reports into one accurate, non-redundant, implementation-ready answer. Call out unresolved assumptions.",
)

Created orchestration-requirements-agent version 1
Created orchestration-implementation-agent version 1
Created orchestration-manager-agent version 1


## Run the orchestration

The two sub-agents work independently. Their reports are then delegated to the orchestrator for synthesis.

In [4]:
user_request = "Design a secure Azure Storage Account deployment using Azure CLI for a production application."

requirements_report = await invoke_agent(requirements_agent, user_request)
implementation_report = await invoke_agent(implementation_agent, user_request)

orchestrator_prompt = f"""Create the final response for this request:
{user_request}

Requirements analyst report:
{requirements_report}

Implementation specialist report:
{implementation_report}

Return a concise architecture summary, ordered implementation steps, Azure CLI commands, security controls, and validation checklist.
"""

final_answer = await invoke_agent(orchestrator_agent, orchestrator_prompt)
display(Markdown("## Requirements Sub-Agent\n" + requirements_report))
display(Markdown("## Implementation Sub-Agent\n" + implementation_report))
display(Markdown("## Orchestrator Final Answer\n" + final_answer))

## Requirements Sub-Agent
### **Goals**  
1. Deploy a secure Azure Storage Account for production use.  
2. Ensure compliance with security best practices (e.g., encryption, restricted access).  
3. Automate the deployment process using Azure CLI.  
4. Optimize for scalability and performance.

---

### **Assumptions**  
1. The production application requires Blob, Queue, or File storage features.  
2. The deployment will use Azure CLI version >= 2.0.  
3. A valid Azure subscription and resource group exist.  
4. Network security rules will need integration with a virtual network or public endpoint protection.  

---

### **Constraints**  
1. Resource naming conventions must follow Azure standards (e.g., alphanumeric, max length 24 for storage account).  
2. Only Azure CLI tools are used; no GUI-based modifications allowed.  
3. Deployment must comply with organizational security policies.  
4. Access must adhere to Azure RBAC or Shared Access Signatures (SAS) for least privilege.  
5. Integration with the production application should not cause downtime.  

---

### **Risks**  
1. Misconfiguration of access policies can lead to unauthorized access.  
2. Potential Azure CLI syntax or script errors during deployment.  
3. Performance degradation if improper storage tiers are selected.  
4. Reduced availability if redundancy options or geo-replication are misconfigured.  
5. Network access misconfigurations could block legitimate application traffic.  

---

### **Acceptance Criteria**  
1. The Azure Storage Account is deployed successfully using CLI commands.  
2. Encryption at rest is enabled (e.g., Azure-managed keys).  
3. Access is restricted via Azure RBAC, private endpoints, or virtual network service endpoints.  
4. Logs for storage access and operations are enabled for monitoring.  
5. The storage tiers and redundancy options align with application needs and budget.  
6. Deployment conforms to organizational security requirements.  

## Implementation Sub-Agent
Below are detailed steps, Azure CLI commands, security best practices, and verification checks for deploying a highly secure Azure Storage Account for a production application:

---

### **1. Plan and Preparation**
Before starting the deployment, understand the production application requirements, such as:
- The region for deployment.
- Performance tier: Standard (magnetic disks) vs. Premium (solid-state drives).
- Access tier: Hot (frequent access), Cool (infrequent access), or Archive (long-term storage).
- Naming conventions for the resource group and storage account.
- Security needs, such as encryption, access control, and firewalls.

---

### **2. Deployment Steps**  
#### **a) Create a Resource Group**
```bash
az group create --name <ResourceGroupName> --location <Region>
```
Example:
```bash
az group create --name ProdAppRG --location eastus
```

#### **b) Create the Storage Account**
Create a secure storage account with advanced security features:
```bash
az storage account create \
  --name <StorageAccountName> \
  --resource-group <ResourceGroupName> \
  --location <Region> \
  --sku Standard_LRS \
  --kind StorageV2 \
  --encryption-services blob,file,table,queue \
  --access-tier Hot \
  --enable-hierarchical-namespace true \
  --allow-blob-public-access false \
  --min-tls-version TLS1_2 \
  --default-action Deny
```
Options:
- `--sku`: Pricing tier (Standard_LRS is locally redundant storage for production).
- `--kind StorageV2`: Use the latest general-purpose v2 storage account.
- `--enable-hierarchical-namespace`: Enable Azure Data Lake Storage Gen2 features (if needed for advanced analytics workloads).
- `--min-tls-version TLS1_2`: Enforce minimum TLS version 1.2.
- `--default-action Deny`: Block all network traffic by default. Add firewall rules later to allow only private or trusted IPs.

Example:
```bash
az storage account create \
  --name prodstorageacct01 \
  --resource-group ProdAppRG \
  --location eastus \
  --sku Standard_LRS \
  --kind StorageV2 \
  --encryption-services blob,file,table,queue \
  --access-tier Hot \
  --enable-hierarchical-namespace true \
  --allow-blob-public-access false \
  --min-tls-version TLS1_2 \
  --default-action Deny
```

---

### **3. Secure Access Controls**
#### **a) Enable Private Endpoint**
Restrict access to the Storage Account by enabling a private endpoint. This ensures traffic only flows via your Azure Virtual Network.
```bash
az network private-endpoint create \
  --name <PrivateEndpointName> \
  --resource-group <ResourceGroupName> \
  --vnet-name <VirtualNetworkName> \
  --subnet <SubnetName> \
  --private-connection-resource-id $(az storage account show --name <StorageAccountName> --resource-group <ResourceGroupName> --query id --output tsv) \
  --group-id blob
```

Example:
```bash
az network private-endpoint create \
  --name prodstorageendpoint \
  --resource-group ProdAppRG \
  --vnet-name prodvnet01 \
  --subnet prodsubnet01 \
  --private-connection-resource-id $(az storage account show --name prodstorageacct01 --resource-group ProdAppRG --query id --output tsv) \
  --group-id blob
```

#### **b) Configure Firewall Rules**
Allow trusted IP ranges or subnets to access the Storage Account.
```bash
az storage account network-rule add \
  --resource-group <ResourceGroupName> \
  --account-name <StorageAccountName> \
  --ip-address <TrustedIPAddress>
```
Example:
```bash
az storage account network-rule add \
  --resource-group ProdAppRG \
  --account-name prodstorageacct01 \
  --ip-address 203.0.113.25
```

#### **c) Disable Public Access**
Block the ability to access blobs directly via public endpoints. This ensures the data is isolated.
```bash
az storage account update \
  --name <StorageAccountName> \
  --resource-group <ResourceGroupName> \
  --allow-blob-public-access false
```

---

### **4. Enable Advanced Security Features**
#### **a) Enable Azure Defender for Storage**
Enable threat detection to secure against malware, unauthorized access, etc.
```bash
az security pricing create \
  --resource-type "StorageAccounts" \
  --tier "Standard"
```

#### **b) Enable Soft Delete (Protection for Blob Data)**
Configure soft delete to allow recovery of accidentally deleted blobs.
```bash
az storage blob-service-properties update \
  --account-name <StorageAccountName> \
  --resource-group <ResourceGroupName> \
  --enable-delete-retention true \
  --delete-retention-days <NumberOfDays>
```
Example:
```bash
az storage blob-service-properties update \
  --account-name prodstorageacct01 \
  --resource-group ProdAppRG \
  --enable-delete-retention true \
  --delete-retention-days 30
```

#### **c) Enable Immutable Blob Storage (Optional)**
Enforce a Write-Once-Read-Many (WORM) policy for regulatory compliance (if required):
```bash
az storage container immutability-policy create \
  --account-name <StorageAccountName> \
  --container-name <ContainerName> \
  --resource-group <ResourceGroupName> \
  --allow-protected-append-writes true \
  --immutability-period-days <NumberOfDays>
```

---

### **5. Verification Checks**
#### **a) Storage Account Properties**
Verify the storage account configuration:
```bash
az storage account show \
  --name <StorageAccountName> \
  --resource-group <ResourceGroupName> \
  --query "{name:name, sku:sku.name, kind:kind, accessTier:accessTier, publicAccess:allowBlobPublicAccess, minTLSVersion:minTlsVersion, defaultAction:networkRuleSet.defaultAction}" \
  --output table
```

#### **b) Private Endpoint Verification**
Check if the private endpoint exists and is linked correctly:
```bash
az network private-endpoint show \
  --name <PrivateEndpointName> \
  --resource-group <ResourceGroupName> \
  --query "{name:name, privateLinkServiceConnectionState:privateLinkServiceConnections[0].privateLinkServiceConnectionState.status}" \
  --output table
```

#### **c) Test Connectivity**
Test blob connectivity via your private endpoint using the Azure Storage Explorer or the `az storage` CLI:
```bash
az storage blob list \
  --account-name <StorageAccountName> \
  --container-name <ContainerName> \
  --auth-mode login
```

#### **d) Check Audit Logs**
Ensure diagnostic settings are configured to route Storage Account logs to Log Analytics or Event Hub for operational monitoring:
```bash
az monitor diagnostic-settings list \
  --resource $(az storage account show --name <StorageAccountName> --resource-group <ResourceGroupName> --query id --output tsv)
```

---

### **6. Maintenance & Monitoring**
- Regularly review access and security rules (`az storage account network-rule list`).
- Monitor logs for unauthorized attempts (`az monitor activity-log list`).
- Enforce naming and resource conventions to organize resources effectively.
- Rotate account keys periodically:
  ```bash
  az storage account keys renew --account-name <StorageAccountName> --key primary
  ```

By following these steps and implementing security controls, the Azure Storage Account deployment will be highly secure and production-ready.

## Orchestrator Final Answer
### **Secure Azure Storage Account Deployment with Azure CLI – Final Implementation Plan**

---

### **Architecture Summary**
1. **Storage Account Type**: General-purpose v2 Storage Account (`StorageV2`) for maximum feature compatibility.
2. **Region**: Choose based on application requirements for lower latency and availability (e.g., `eastus`).
3. **Redundancy**: Use locally redundant storage (`Standard_LRS`) for high availability; revise if geo-replication is needed.
4. **Access Control**: Enforce private endpoints and firewalls, disable public access.
5. **Encryption**: Enable encryption at rest with Microsoft-managed keys.
6. **Additional Features**: Enable Azure Defender for Storage, soft delete for accidental recovery, and diagnostic logs for monitoring.

---

### **Ordered Implementation Steps**

#### **Step 1 – Create Resource Group**
```bash
az group create --name ProdAppRG --location eastus
```

#### **Step 2 – Deploy Secure Storage Account**
```bash
az storage account create \
  --name prodstorageacct01 \
  --resource-group ProdAppRG \
  --location eastus \
  --sku Standard_LRS \
  --kind StorageV2 \
  --encryption-services blob,file,table,queue \
  --access-tier Hot \
  --enable-hierarchical-namespace true \
  --allow-blob-public-access false \
  --min-tls-version TLS1_2 \
  --default-action Deny
```

#### **Step 3 – Configure Private Endpoint**
```bash
az network private-endpoint create \
  --name prodstorageendpoint \
  --resource-group ProdAppRG \
  --vnet-name prodvnet01 \
  --subnet prodsubnet01 \
  --private-connection-resource-id $(az storage account show --name prodstorageacct01 --resource-group ProdAppRG --query id --output tsv) \
  --group-id blob
```

#### **Step 4 – Add Firewall Rules for Trusted IPs**
```bash
az storage account network-rule add \
  --resource-group ProdAppRG \
  --account-name prodstorageacct01 \
  --ip-address 203.0.113.25
```

#### **Step 5 – Enable Azure Defender for Storage**
```bash
az security pricing create --resource-type "StorageAccounts" --tier "Standard"
```

#### **Step 6 – Enable Soft Delete**
```bash
az storage blob-service-properties update \
  --account-name prodstorageacct01 \
  --resource-group ProdAppRG \
  --enable-delete-retention true \
  --delete-retention-days 30
```

#### **Step 7 – Enable Diagnostic Settings**
Route logs and metrics to a Log Analytics workspace or Event Hub.
```bash
az monitor diagnostic-settings create \
  --name StorageDiagSettings \
  --resource $(az storage account show --name prodstorageacct01 --resource-group ProdAppRG --query id --output tsv) \
  --workspace <LogAnalyticsResourceId> \
  --metrics '[{"category": "AllMetrics", "enabled": true}]' \
  --logs '[{"category": "StorageRead", "enabled": true}, {"category": "StorageWrite", "enabled": true}, {"category": "StorageDelete", "enabled": true}]'
```

#### **Step 8 – Test Connectivity**
Verify restricted blob access via private endpoint.
```bash
az storage blob list \
  --account-name prodstorageacct01 \
  --container-name testcontainer \
  --auth-mode login
```

#### **Step 9 – Rotate Keys Periodically**
Rotate keys securely every predefined period:
```bash
az storage account keys renew --account-name prodstorageacct01 --key primary
```

---

### **Security Controls**

1. **Private Endpoints**: Isolate traffic within an Azure Virtual Network.
2. **Firewall Rules**: Restrict access to trusted IPs.
3. **TLS Enforcement**: Set minimum TLS version to `TLS1_2`.
4. **Disable Public Access**: Completely block anonymous access.
5. **Azure Defender**: Enable threat detection to monitor suspicious activities.
6. **Soft Delete**: Provide accidental blob recovery for up to 30 days.
7. **Immutable Storage (Optional)**: Implement Write-Once-Read-Many (WORM) storage for compliance.

---

### **Validation Checklist**

- **Storage Account Properties**:
  ```bash
  az storage account show \
    --name prodstorageacct01 \
    --resource-group ProdAppRG \
    --query "{name:name, sku:sku.name, kind:kind, accessTier:accessTier, publicAccess:allowBlobPublicAccess, minTLSVersion:minTlsVersion, defaultAction:networkRuleSet.defaultAction}" \
    --output table
  ```

- **Firewall and Network Rules**:
  ```bash
  az storage account network-rule list --account-name prodstorageacct01 --resource-group ProdAppRG
  ```

- **Private Endpoint Status**:
  ```bash
  az network private-endpoint show \
    --name prodstorageendpoint \
    --resource-group ProdAppRG \
    --query "{name:name, privateLinkServiceConnectionState:privateLinkServiceConnections[0].privateLinkServiceConnectionState.status}" \
    --output table
  ```

- **Monitor Logs**:
  ```bash
  az monitor activity-log list --resource $(az storage account show --name prodstorageacct01 --resource-group ProdAppRG --query id --output tsv)
  ```

---

### **Unresolved Assumptions**

1. **Application-Specific Needs**: Verify if advanced features like immutable blob storage or geo-replication are required.
2. **Network Configuration**: VNet and subnet details for the private endpoint are assumed to exist but must be explicitly validated.
3. **Backup and Recovery**: No explicit commands include backup service integrations; additional planning may be necessary.

This plan ensures secure, scalable Azure Storage Account deployment for production while adhering to industry and organizational best practices.

## Close clients

In [5]:
await project_client.close()
await credential.close()
print("Clients closed.")

Clients closed.
